# Stitching together `.cha` files from the checkpoints

Perhaps inadvertently, I've saved the outputs for each conversation as a CEDA checkpoint. We need to iterate through them and stitch together the final file for analysis.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
DATA_PATH = 'data'

INPUT_PATH = os.path.join(DATA_PATH, 'chas-ckpts')

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

OUTPUT_NAME = os.path.join(OUTPUT_PATH, 'cha-{}-ceda-analysis.tsv')

In [3]:
file_ending = 'ko_corpus.tsv.pt'

In [4]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

## Grabbing each file and creating a dataframe for it

In [5]:
files = [
    f for f in grab_all_files(DATA_PATH, file_ending) 
    if (not f.startswith('.')) 
       # and ('cha_corpus' in f)
]
print(files)

['data/ckpts/graph-obj-k_callfriend-ko_corpus.tsv.pt']


In [6]:
for f in tqdm(files):
    
    output_name_ = str(OUTPUT_NAME).format(f.split('-')[-1].replace('.tsv.pt', ''))
    
    print(f, output_name_)
    
    ckpt = torch.load(f)
    df = pd.DataFrame(ckpt['meta_data'])
    
    if 'raw_text' in list(df):
        print('hyp confirmed! raw_text present.')
        del df['raw_text']
    
    # number of tokens per text
    df['nx'] = ckpt['N'][:,0].tolist()
    df['ny'] = ckpt['N'][:,1].tolist()
    
    # CE values
    df['Hxy'] = ckpt['M'][:,0].tolist()
    df['Hyx'] = ckpt['M'][:,1].tolist()
    
    print(df.shape)
    
    df.to_csv(
        output_name_,
        index=False,
        encoding='utf-8',
        sep='\t'
    )
    

  0%|          | 0/1 [00:00<?, ?it/s]

data/ckpts/graph-obj-k_callfriend-ko_corpus.tsv.pt data/lme_ready/cha-ko_corpus-ceda-analysis.tsv
(2197289, 13)
